# Test Accuracy per Attack per Algorithm

In [1]:
%load_ext autoreload
import sys
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [3]:
%autoreload 2
from lib.graph_factory import GraphFactory, add_graph_plot
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed
from lib.classifier import ByzClassifier
from lib.system import SystemSimulator
from lib.config import BASE_CONF, NUM_NODES

In [4]:
# Helper Functions
fmt = lambda sec : f"{sec // 60}m {sec % 60:.2f}s"

In [5]:
RUN_DIR = Path().resolve()
config = BASE_CONF
b = config['train']['b']
config['graph_args'] = {
    'ws_k': 6,
    'ws_p': 0.1,
    'rand_reg_deg':b+1,
    'geom_radius':0.4,
    'er_p':min(1.0, 3.0 * np.log(NUM_NODES) / NUM_NODES),
}
config['graph_weights'] = 'MH'
graphs = ['random-regular','watts-strogatz', 'geometric']
ATKS = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
ALGS = ['RDSGD', 'IOS', 'SCC', 'TriMean', 'CooMed']
COLORS = dict(zip(ALGS, ['C0', 'C1', 'C2', 'C3', 'C4']))
SEEDS = [777, 111, 555, 444, 888]

In [6]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], b)
proj_const_estimator = ProjConstEstimator(config, global_dataset, gf) 
clf = ByzClassifier(config, global_dataset, gf)
sys_sim = SystemSimulator(config, global_dataset, gf)

In [ ]:
test_acc_summaries = dict()
for g in graphs:
    config['graph_type']=g
    print(f"{g} graph")
    
    prelim_metrics = dict()
    proj_const_estimator.configure(config, seed('proj-const-estimator'))
    proj_const = proj_const_estimator.estimate()

    t0 = time.perf_counter()
    clf_sim_data = clf.run_simulations(config, proj_const, seed('clf','simulation-seed'))
    print(f"clf sim took {fmt(time.perf_counter() - t0)}")

    t1 = time.perf_counter()
    clf_est, clf_preproc = clf.fit()
    print(f"clf fit took {fmt(time.perf_counter() - t1)}")
    
    clf_op_pt = clf.calc_optimal_op_pt()
    clf_metrics = clf.test()
    
    prelim_metrics['proj_const'] = proj_const
    prelim_metrics.update(clf_op_pt)
    prelim_metrics.update(clf_metrics)
    
    display(pd.DataFrame([prelim_metrics]))  

    plot = plt.subplots(2,3,figsize=(12,8))
    rows = []
    for atk_idx, atk in enumerate(ATKS):
        for s in SEEDS:
            config['sys']['atk_type'] = atk
            sim_params = {
                'algorithms': ALGS, 
                'atk_type': atk,
                'threat_model': 'T3', 
                'seed': s
            }
            res = {'timestamp': pd.to_datetime(datetime.now()), **prelim_metrics}
            
            sys_sim.init_simulation(config, proj_const, clf_preproc, clf_est, clf_op_pt, 
                                    seed(g,atk,s), is_printing_logs=False)
    
            t3 = time.perf_counter()
            df = sys_sim.simulate(sim_params)
            print(f"system sim ({atk} attack, seed {s}) took {fmt(time.perf_counter() - t3)}")
    
            for alg in ALGS:
                if(alg == 'RDSGD'):
                    # ONLY TRUE FOR MH WEIGHTS!!
                    assert(df.loc[alg, 'test_acc_pi'].iloc[-1] == df.loc[alg, 'test_acc'].iloc[-1])
                    test_acc = df.loc[alg, 'test_acc_pi'].iloc[-1]
                else:
                    test_acc = df.loc[alg, 'test_acc'].iloc[-1]
                payload = {
                    'alg':alg,
                    'atk':atk,
                    'seed':s,
                    'test_acc': test_acc,
                }
                if s == SEEDS[0]: display(alg, pd.DataFrame([payload]))
                rows.append(payload)
                
    df_summary = pd.DataFrame(rows).groupby(['atk', 'alg'])['test_acc'].agg(['mean', 'std']).unstack('alg')
    test_acc_summaries[g] = df_summary